# POC — Veridion: Entity Resolution & Data Quality

This notebook runs the main steps: load config & CSV, run ER, perform QC, export Power BI-ready CSVs, and generate a PDF report.

In [11]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for rel in ('data/processed', 'outputs/powerbi', 'outputs/pdf'):
    (PROJECT_ROOT / rel).mkdir(parents=True, exist_ok=True)
PROJECT_ROOT


WindowsPath('D:/Veridion_Nr_5/Popescu_Vlad_Veridion_Challenge_Nr_5')

In [12]:
import pandas as pd
import yaml
from pathlib import Path
from src.veridion_poc.config import load_config

# Update config with absolute path
config_path = PROJECT_ROOT / 'config/config.yaml'
with open(config_path) as f:
    raw_config = yaml.safe_load(f)
raw_config['input']['path'] = str(PROJECT_ROOT / raw_config['input']['path'])

# Save temporary config
tmp_config = config_path.with_name('config_tmp.yaml')
with open(tmp_config, 'w') as f:
    yaml.dump(raw_config, f)

# Load config and clean up
cfg = load_config(str(tmp_config))
tmp_config.unlink()

# Load data
df = pd.read_csv(PROJECT_ROOT / 'data/raw/presales_data_sample (1).csv')
print(f"Loaded {len(df):,} rows")
df.head()

Loaded 2,951 rows


,input_row_key,input_company_name,input_main_country_code,input_main_country,input_main_region,input_main_city,input_main_postcode,input_main_street,input_main_street_number,veridion_id,...,twitter_url,instagram_url,linkedin_url,ios_app_url,android_app_url,youtube_url,tiktok_url,technologies,created_at,last_updated_at
0,0,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,PK,Pakistan,Sindh,Karachi,NaN,NaN,NaN,26e22210-93e5-11eb-b997-8dd98d09cf25,...,NaN,NaN,http://www.linkedin.com/company/mnet-services-...,NaN,NaN,NaN,NaN,web servers: apache http server - 2 | javascri...,2020-02-25T14:47:51.000Z,2024-11-29T04:18:00.109Z
1,0,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,PK,Pakistan,Sindh,Karachi,NaN,NaN,NaN,01004641-1dd8-11ef-9268-316fc8e174dd,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-29T16:24:11.019Z,2025-04-20T15:03:24.026Z
2,0,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,PK,Pakistan,Sindh,Karachi,NaN,NaN,NaN,8266efc1-13e7-11ec-aa14-7bf90e1e10f1,...,https://twitter.com/Network24seven%20,NaN,https://www.linkedin.com/company/51633612%20%20,NaN,NaN,NaN,NaN,NaN,2021-09-12T05:28:48.000Z,2025-03-18T23:08:37.059Z
3,0,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,PK,Pakistan,Sindh,Karachi,NaN,NaN,NaN,0183f0b2-93e5-11eb-be5a-4f810ab55f2e,...,https://twitter.com/emeriosoft,NaN,https://www.linkedin.com/company/emeriosoft,NaN,NaN,NaN,NaN,miscellaneous: popper | maps: google maps | pr...,2020-05-03T12:33:22.000Z,2025-03-31T16:16:58.462Z
4,0,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,PK,Pakistan,Sindh,Karachi,NaN,NaN,NaN,87bb7cde-93e4-11eb-8474-3bbe2d07207d,...,https://twitter.com/AsiaticPR,https://www.instagram.com/asiaticpublicrelations/,https://www.linkedin.com/company/asiaticpublic...,NaN,NaN,https://www.youtube.com/channel/UClKgHvHIuOu4K...,NaN,miscellaneous: babel | javascript libraries: c...,2020-02-19T03:58:25.000Z,2024-11-25T11:35:47.963Z


In [5]:
from src.veridion_poc.er import entity_resolution

er_df = entity_resolution(df, cfg)
er_df.head()


,row_index,status,score,best_cand_index,best_cand_name,best_cand_country,best_cand_id
0,0,accept,0.7167,1,New Millennium Network Private Limited | MNET ...,Pakistan,26e22210-93e5-11eb-b997-8dd98d09cf25
1,1,maybe,0.5591,2,Private Helipad Ali Villa,Pakistan,01004641-1dd8-11ef-9268-316fc8e174dd
2,2,maybe,0.5926,2,24seven Research Network,India,8266efc1-13e7-11ec-aa14-7bf90e1e10f1
3,3,maybe,0.5700,2,Emeriosoft | EMERIOSOFT Private L.T.D.,Pakistan,0183f0b2-93e5-11eb-be5a-4f810ab55f2e
4,4,accept,0.7167,1,Asiatic Public Relations Network Private Limited,Pakistan,87bb7cde-93e4-11eb-8474-3bbe2d07207d


In [6]:
from src.veridion_poc.qc import detect_duplicates, missingness

qc_columns = [cfg.input.name_column]
if cfg.input.country_column:
    qc_columns.append(cfg.input.country_column)
dup_df = detect_duplicates(df, cfg)
miss_df = missingness(df, qc_columns)
dup_df.head(), miss_df.head()


(                                               key  \
 0         apple distribution international|ireland   
 1         digi telecommunications sdn bhd|malaysia   
 2                      eci media management|sweden   
 3  tata communications international pte|singapore   
 4          24 seven media network private|pakistan   
 
                                          row_indices  count  
 0  [1886, 1887, 1888, 1889, 1890, 1941, 1942, 194...     10  
 1  [235, 236, 237, 238, 239, 240, 241, 242, 243, ...     10  
 2  [315, 316, 317, 318, 319, 320, 321, 322, 323, ...     10  
 3  [2191, 2192, 2193, 2194, 2195, 2196, 2197, 219...     10  
 4                                    [0, 1, 2, 3, 4]      5  ,
                column  missing  missing_pct
 0  input_company_name        0          0.0
 1  input_main_country        0          0.0)

In [7]:
processed_dir = PROJECT_ROOT / 'data/processed'
pbi_dir = PROJECT_ROOT / 'outputs/powerbi'
pdf_dir = PROJECT_ROOT / 'outputs/pdf'

er_df.to_csv(processed_dir / 'entity_resolution.csv', index=False)
er_df.to_csv(pbi_dir / 'fact_entity_resolution.csv', index=False)
dup_df.to_csv(processed_dir / 'duplicates.csv', index=False)
miss_df.to_csv(processed_dir / 'missingness.csv', index=False)
'Exported CSVs'


'Exported CSVs'

In [8]:
from scripts.run_er import _build_sections
from src.veridion_poc.report_pdf import generate_pdf

sections = _build_sections(cfg, er_df, dup_df, miss_df)
pdf_path = pdf_dir / 'POC_Veridion_ER_QC.pdf'
generate_pdf(str(pdf_path), 'POC Report – Veridion (ER + QC)', sections)
pdf_path


WindowsPath('D:/Veridion_Nr_5/Popescu_Vlad_Veridion_Challenge_Nr_5/outputs/pdf/POC_Veridion_ER_QC.pdf')

In [10]:
# Show summary of Entity Resolution results
print("DataFrame columns:", er_df.columns.tolist())

resolution_counts = er_df['status'].value_counts()
total = len(er_df)

print("\nEntity Resolution Results:")
print("-" * 30)
for status, count in resolution_counts.items():
    percentage = count / total * 100
    print(f"{status}: {count:,} ({percentage:.1f}%)")

print(f"\nTotal records processed: {total:,}")

DataFrame columns: ['row_index', 'status', 'score', 'best_cand_index', 'best_cand_name', 'best_cand_country', 'best_cand_id']

Entity Resolution Results:
------------------------------
accept: 2,124 (72.0%)
maybe: 460 (15.6%)
unmatched: 367 (12.4%)

Total records processed: 2,951
